# OpenAnalytics Behavioral Machine Learning Engine
### End-to-End Walkthrough: Feature Engineering, Propensity Modeling & Sub-Microsecond Export

This notebook demonstrates:
1. **Clickstream Data Generation & Ingestion**: Modeling realistic e-commerce shopper archetypes.
2. **Feature Extraction**: Turning raw clickstream events into high-signal behavioral vectors.
3. **Model Training**: Multi-intent modeling for **Purchase Intent**, **Session Churn**, and **Price Sensitivity**.
4. **Model Explainability**: Local feature attributions (SHAP-style) explaining why shoppers convert or drop off.
5. **Zero-Overhead Export**: Calibrated JSON weight serialization for nanosecond Go engine inference.

In [ ]:
import sys
from pathlib import Path

# Ensure project imports work from notebooks/
sys.path.insert(0, str(Path.cwd().parent))

from pipeline.synthetic_generator import generate_ecommerce_dataset
from pipeline.feature_engineering import extract_features, journey_to_features
from models.registry import registry
from inference.engine import InferenceEngine

## 1. Simulate E-Commerce Shopper Journeys
We generate 5,000 shopper journeys across 5 realistic behavioral archetypes:
- **Quick Bounce**: Fast exit (<30s dwell, 1-2 views)
- **Casual Explorer**: Broad catalog browsing, low carting
- **Price Hunter**: Discount focused, high category switching
- **Cart Abandoner**: Adds products to cart, drops off at checkout
- **Decisive Buyer**: Focused product views, adds to cart, purchases

In [ ]:
dataset = generate_ecommerce_dataset(n_samples=5000, seed=42)
print(f"Generated {len(dataset)} sessions.")
print("Sample shopper:", dataset[0])

## 2. Feature Extraction & Engineering

In [ ]:
sample_features = journey_to_features(dataset[0])
for k, v in sample_features.items():
    print(f"{k:25s}: {v}")

## 3. Real-Time Multi-Model Scoring & Explainability

In [ ]:
engine = InferenceEngine()

# High-intent scenario
high_intent_shopper = {
    "views_count": 8,
    "cart_adds_count": 3,
    "distinct_products": 2,
    "dwell_time_seconds": 450,
    "avg_scroll_depth": 0.85,
    "sale_view_ratio": 0.15,
}

scores = engine.score_session(high_intent_shopper)
print("Multi-Model Scores:", scores)

explanation = engine.explain("cart_intent", high_intent_shopper)
print("\nPurchase Intent Drivers:")
for driver in explanation["top_drivers"]:
    feat_name = driver[0]
    impact = driver[1]["impact"]
    print(f" - {feat_name}: {impact:+.4f} logit impact")